# Demo 1 - Token Limits & Quota Enforcement (Azure API Management)

## Scenario & talk track

AI workloads are billed and rate-limited by **tokens**, not requests, which
makes traditional "requests per second" throttling insufficient for
governing LLM traffic. This demo shows how Azure API Management (APIM) can
enforce **token-aware governance** in front of Azure OpenAI, covering three
concerns that matter to any organization running shared AI infrastructure:

- **Cost control** -- cap how many tokens a consumer can burn, independent of
  how large or small their individual prompts/completions are.
- **Noisy-neighbor protection** -- a single caller bursting requests should
  not be able to starve token capacity for everyone else sharing the
  deployment.
- **Per-consumer fairness / daily budgets** -- beyond a short per-minute
  limit, a daily token budget prevents slow-but-steady overuse across a day.

We will apply an APIM policy that:

1. Enforces a **tokens-per-minute (TPM)** limit using the
   `azure-openai-token-limit` policy, surfacing `tokens-consumed` and
   `remaining-tokens` response headers plus a `Retry-After` header when
   the limit is exceeded (HTTP 429).
2. Enforces a small **daily token budget** on top of that, returning HTTP
   403 with a clear explanation once the cumulative daily cap is exceeded.

Everything is created and configured **before** we make a single call to the
API -- that is deliberate, and mirrors how you'd roll this out for a real
workload.


## Demo isolation

Because this is a shared workshop APIM instance, we need this demo's traffic
counters to be **completely isolated** from:

- other attendees / other runs of this same demo,
- the other three demos in this workshop,
- any real production traffic on the same APIM instance.

We achieve isolation two ways:

1. **A dedicated APIM subscription** (`demo1-token-governance-sub`) created
   just for this demo. All calls in this notebook use this subscription's
   key, so its counters never mix with anything else.
2. **A unique `x-demo-run` suffix** baked directly into the policy's
   `counter-key` expression:
   `context.Subscription.Id + "-" + context.Request.Headers.GetValueOrDefault("x-demo-run","default")`.
   Every request we send includes an `x-demo-run` header carrying the
   current `DEMO_RUN` value. Because the counter key includes this value,
   **regenerating `DEMO_RUN` and re-applying the header instantly resets the
   demo** -- no need to wait for the TPM window or the daily window to roll
   over. This is exactly what the "Reset" section near the end of this
   notebook does.

`DEMO_RUN` defaults to a short random suffix and is persisted to `.env` so
reruns are stable until you explicitly reset.


In [ ]:
import sys
sys.path.append("..")

import time
import uuid

from shared import auth, config, apim, display

cfg = config.load_config(interactive=True)
config.validate_config(cfg)

DEMO_API_ID = "demo1-openai-api"
DEMO_BACKEND_ID = "demo1-openai-backend"
DEMO_PRODUCT_ID = "demo1-token-governance-product"
DEMO_SUBSCRIPTION_ID = "demo1-token-governance-sub"
DEMO_NAMED_VALUE_KEY = "demo1-aoai-key"
DEMO_NAMED_VALUE_TPM = "demo1-tokens-per-minute"
DEMO_NAMED_VALUE_DAILY_CAP = "demo1-daily-token-cap"

TOKENS_PER_MINUTE = 200          # deliberately small so a burst trips it easily
DAILY_TOKEN_CAP = 400            # deliberately small so it's reachable in a workshop

DEMO_RUN = cfg.demo_run
# "v1" targets the Microsoft Foundry surface (/openai/v1/chat/completions,
# deployment passed in the body as "model"); "classic" targets the Azure
# OpenAI surface (/openai/deployments/{deployment}/chat/completions?api-version=...).
API_STYLE = cfg.aoai_api_style

print(f"Demo isolation subscription id: {DEMO_SUBSCRIPTION_ID}")
print(f"Azure OpenAI endpoint: {cfg.aoai_endpoint} (api style: {API_STYLE})")
print(f"Current DEMO_RUN suffix: {DEMO_RUN}")


## Create the AI API (idempotent)

We now create everything Demo 1 needs on the APIM instance:

- A **backend** pointing at the Azure OpenAI / Microsoft Foundry resource
  root. The API policy references this backend with `set-backend-service`, so
  the backend entity is the authoritative origin for governed traffic.
- An **API** at path `/demo1-openai`. Its `serviceUrl` is still set to the
  same resource root as a harmless fallback for requests that bypass the
  policy or if policy application fails, but the backend entity drives routing
  once the policy below is applied.
- A **named value** holding the Azure OpenAI key (used as a fallback). We
  prefer **managed identity** (`authentication-managed-identity`) for the
  backend-to-AOAI call; if no key is supplied and managed identity is not
  yet configured on the APIM instance, calls will fail with a clear error
  and instructions to either supply a key or grant the APIM system-assigned
  identity the `Cognitive Services OpenAI User` role on the Azure OpenAI
  resource.
- The `chat/completions` **operation**.
- A dedicated **product** and **subscription** for isolation (see above).
- Named values for the TPM limit and daily token cap, referenced by the
  policy so they're easy to tune without editing XML.

The backend must be created before the policy is applied because APIM validates
`set-backend-service` references when the policy is saved. Every call below
uses PUT/upsert semantics, so re-running this notebook does not fail or
duplicate resources.


In [ ]:
display.header("Creating backend")
backend = apim.ensure_backend(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    backend_id=DEMO_BACKEND_ID,
    backend_url=cfg.aoai_endpoint,
    description="Demo 1 Azure OpenAI backend",
    # APIM backend protocol is an enum: "http" means HTTP-style backend
    # (vs "soap"). Keep HTTPS origins as protocol="http"; TLS transport
    # comes from cfg.aoai_endpoint's https:// URL scheme, and "https" is
    # not a valid ARM value here.
    protocol="http",
)
display.banner(f"Backend '{DEMO_BACKEND_ID}' ensured.", kind="success")


In [ ]:
display.header("Creating API")
api = apim.ensure_api(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    display_name="Demo 1 - Token Governance (Azure OpenAI)",
    path="demo1-openai",
    service_url=cfg.aoai_endpoint.rstrip("/"),
    subscription_required=True,
)
display.banner(f"API '{DEMO_API_ID}' ensured at path /demo1-openai.", kind="success")


In [ ]:
display.header("Creating chat/completions operation")

if API_STYLE == "v1":
    OPERATION_URL_TEMPLATE = "/openai/v1/chat/completions"
else:
    OPERATION_URL_TEMPLATE = f"/openai/deployments/{cfg.aoai_deployment}/chat/completions"

operation = apim.ensure_operation(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    operation_id="chat-completions",
    display_name="Chat Completions",
    method="POST",
    url_template=OPERATION_URL_TEMPLATE,
)
display.banner(f"Operation 'chat-completions' ensured at {OPERATION_URL_TEMPLATE}.", kind="success")


In [ ]:
display.header("Creating named values")

if cfg.aoai_key:
    apim.ensure_named_value(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        named_value_id=DEMO_NAMED_VALUE_KEY,
        display_name=DEMO_NAMED_VALUE_KEY,
        value=cfg.aoai_key,
        secret=True,
    )
    display.banner("AOAI key named value ensured (key masked).", kind="success")
else:
    display.banner(
        "No AOAI key supplied -- assuming managed identity is configured for "
        "this APIM instance to call Azure OpenAI. If calls fail with 401/403, "
        "grant the APIM system-assigned identity the 'Cognitive Services OpenAI "
        "User' role on the Azure OpenAI resource, or set AOAI_KEY in .env.",
        kind="warning",
    )

apim.ensure_named_value(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    named_value_id=DEMO_NAMED_VALUE_TPM,
    display_name=DEMO_NAMED_VALUE_TPM,
    value=str(TOKENS_PER_MINUTE),
    secret=False,
)
apim.ensure_named_value(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    named_value_id=DEMO_NAMED_VALUE_DAILY_CAP,
    display_name=DEMO_NAMED_VALUE_DAILY_CAP,
    value=str(DAILY_TOKEN_CAP),
    secret=False,
)
display.banner("TPM and daily-cap named values ensured.", kind="success")


In [ ]:
print(f"Sub: {cfg.subscription_id}, RG: {cfg.resource_group}, APIM: {cfg.apim_name}")
print(f"Product ID: {DEMO_PRODUCT_ID}, API ID: {DEMO_API_ID}")

display.header("Creating product and demo-isolated subscription")

apim.ensure_product(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    product_id=DEMO_PRODUCT_ID,
    display_name="Demo1-Token-Governance",
    description="Isolated product for the Demo 1 token-limit workshop scenario.",
    subscription_required=True,
    state="published",
)
apim.ensure_product_api_link(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    product_id=DEMO_PRODUCT_ID,
    api_id=DEMO_API_ID,
)
apim.ensure_subscription(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    apim_subscription_id=DEMO_SUBSCRIPTION_ID,
    display_name="Demo1-Token-Governance-Subscription",
    scope=f"/products/{DEMO_PRODUCT_ID}",
)
display.banner(
    f"Product '{DEMO_PRODUCT_ID}' and subscription '{DEMO_SUBSCRIPTION_ID}' ensured.",
    kind="success",
)


## Apply the policy at API scope (before any calls)

Below is the full policy XML from `policies/demo1-token-limit.xml`. It
combines:

- `set-backend-service` for `demo1-openai-backend`, making the APIM backend
  entity the authoritative Azure OpenAI / Microsoft Foundry origin for
  governed requests.
- `azure-openai-token-limit` for the tokens-per-minute limit, with the
  demo-isolated `counter-key`, `estimate-prompt-tokens="false"`, a
  `retry-after-header-name`, and both token headers enabled.
- A daily quota dimension implemented with `cache-lookup-value` /
  `cache-store-value` plus a `return-response` that issues **403** once the
  configured daily cap is exceeded.

We apply it **at API scope** via an idempotent ARM REST `PUT` to
`.../apis/{apiId}/policies/policy`, and we do this **after** the backend has
been created but **before** any data-plane call is made.


In [ ]:
import re

with open("../policies/demo1-token-limit.xml", encoding="utf-8-sig") as f:
    policy_xml = f.read()

# The policy authenticates to Azure OpenAI with the APIM managed identity by
# default (Foundry resources commonly have key auth disabled). If a key *was*
# supplied, swap that block for an 'api-key' header sourced from the
# demo1-aoai-key named value.
_MI_AUTH_BLOCK = re.compile(
    r"\s*<authentication-managed-identity\b.*?</set-header>", re.DOTALL
)
_API_KEY_BLOCK = (
    '\n    <set-header name="api-key" exists-action="override">'
    "\n      <value>{{demo1-aoai-key}}</value>"
    "\n    </set-header>"
)

if cfg.aoai_key:
    policy_xml, replaced = _MI_AUTH_BLOCK.subn(_API_KEY_BLOCK, policy_xml, count=1)
    if not replaced:
        raise ValueError("Could not find the managed-identity block in the policy XML.")
    display.banner(
        "AOAI key supplied -- the policy will authenticate to the backend with "
        "the 'api-key' header (value from the demo1-aoai-key named value).",
        kind="info",
    )
else:
    display.banner(
        "No AOAI key -- the policy authenticates to the backend with the APIM "
        "managed identity. Grant that identity the 'Cognitive Services OpenAI "
        "User' role on the Azure OpenAI / Foundry resource.",
        kind="info",
    )

print(policy_xml)


In [ ]:
apim.set_api_policy(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    policy_xml=policy_xml,
)
display.banner("Policy applied at API scope.", kind="success")


In [ ]:
applied_policy = apim.get_api_policy(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
)
applied_policy_xml = applied_policy.get("properties", {}).get("value", "")
backend_directive_pattern = (
    r"<set-backend-service\b[^>]*backend-id=[\"']"
    + re.escape(DEMO_BACKEND_ID)
    + r"[\"']"
)

if re.search(backend_directive_pattern, applied_policy_xml):
    display.banner(
        f"Verified applied policy routes through backend '{DEMO_BACKEND_ID}'.",
        kind="success",
    )
else:
    display.banner(
        f"Applied policy is missing set-backend-service for '{DEMO_BACKEND_ID}'.",
        kind="error",
    )
    raise RuntimeError("Applied Demo 1 policy does not select the APIM backend entity.")


### Data-plane call helper

Before making any calls, set up a small helper that calls the gateway with
the demo-isolated subscription key and the current `x-demo-run` header, and
captures status, headers, latency, and (if present) the model reply.


In [ ]:
import requests

GATEWAY_URL = apim.get_gateway_url(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
SUBSCRIPTION_KEY = apim.get_subscription_key(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID
)

print(f"Gateway URL: {GATEWAY_URL}")
print(f"Subscription key: {auth.mask_secret(SUBSCRIPTION_KEY)}")


def call_chat_completion(prompt: str, demo_run: str, max_tokens: int = 50):
    """Call the Demo 1 chat/completions operation and capture governance headers."""
    headers = {
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        "Content-Type": "application/json",
        "x-demo-run": demo_run,
    }

    start = time.time()
    if API_STYLE == "v1":
        # Microsoft Foundry v1 surface: deployment goes in the body as "model"
        # and there is no api-version query parameter.
        url = f"{GATEWAY_URL}/demo1-openai/openai/v1/chat/completions"
        body = {
            "model": cfg.aoai_deployment,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        }
        response = requests.post(url, headers=headers, json=body, timeout=60)
    else:
        url = (
            f"{GATEWAY_URL}/demo1-openai/openai/deployments/"
            f"{cfg.aoai_deployment}/chat/completions"
        )
        body = {
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        }
        response = requests.post(
            url, headers=headers, params={"api-version": cfg.aoai_api_version},
            json=body, timeout=60,
        )
    latency_ms = round((time.time() - start) * 1000, 1)

    reply = None
    if response.status_code == 200:
        try:
            reply = response.json()["choices"][0]["message"]["content"]
        except Exception:
            reply = None

    return {
        "status": response.status_code,
        "latency_ms": latency_ms,
        "tokens_consumed": response.headers.get("tokens-consumed"),
        "remaining_tokens": response.headers.get("remaining-tokens"),
        "retry_after": response.headers.get("Retry-After"),
        "reply": reply,
        "body": response.text,
        "url": response.url,
    }


def explain_failure(result):
    """Print a diagnostic banner for a non-200 data-plane response."""
    status = result["status"]
    display.banner(f"Call did not return 200 (status {status}): {result['body']}", kind="error")

    if status == 404:
        print(f"Requested URL   : {result.get('url')}")
        print(f"AOAI endpoint   : {cfg.aoai_endpoint}")
        print(f"AOAI deployment : {cfg.aoai_deployment}")
        print(f"API style       : {API_STYLE}")
        print(f"API version     : {cfg.aoai_api_version} (classic style only)")
        print(
            "Likely causes: AOAI_ENDPOINT includes a path (it must be the resource "
            "root), AOAI_DEPLOYMENT does not match a deployment on that resource, "
            "or AOAI_API_STYLE is wrong ('v1' for Microsoft Foundry, 'classic' for "
            "Azure OpenAI)."
        )
    elif status in (401, 403):
        print(
            "Likely cause: the APIM managed identity is missing the 'Cognitive "
            "Services OpenAI User' role on the Azure OpenAI / Foundry resource "
            "(or the supplied AOAI_KEY is invalid). Note that a 403 can also be "
            "this demo's daily token cap -- check the response body above."
        )


## Baseline call + read the token headers

A single chat completion, well within the TPM limit, so we can see the
governance headers on a normal, successful call.

- **`tokens-consumed`** -- tokens the policy counted against this request
  (prompt + completion tokens, since `estimate-prompt-tokens="false"` uses
  the actual usage reported by Azure OpenAI).
- **`remaining-tokens`** -- tokens left in the current TPM window for this
  demo-isolated counter key.


In [ ]:
baseline = call_chat_completion("Say hello in one short sentence.", DEMO_RUN)

display.show_table([{
    "status": baseline["status"],
    "latency_ms": baseline["latency_ms"],
    "tokens_consumed": baseline["tokens_consumed"],
    "remaining_tokens": baseline["remaining_tokens"],
    "reply": baseline["reply"],
}])

if baseline["status"] != 200:
    explain_failure(baseline)
else:
    display.banner("Baseline call succeeded -- governance headers captured above.", kind="success")


## Burst to 429 with Retry-After

Now we fire rapid sequential requests until the TPM limit trips a **429**.
We record a per-request timeline (status, tokens consumed, remaining
tokens, `Retry-After`) and plot remaining tokens across the burst. The loop
is bounded so it can't hang a workshop.


In [ ]:
MAX_BURST_REQUESTS = 20

burst_log = []
tripped_at = None

for i in range(1, MAX_BURST_REQUESTS + 1):
    result = call_chat_completion("Count from 1 to 5.", DEMO_RUN, max_tokens=30)
    entry = {
        "request_number": i,
        "status": result["status"],
        "tokens_consumed": result["tokens_consumed"],
        "remaining_tokens": result["remaining_tokens"],
        "retry_after": result["retry_after"],
    }
    burst_log.append(entry)

    if result["status"] == 429:
        tripped_at = i
        display.banner(
            f"429 observed at request #{i}. Retry-After: {result['retry_after']} seconds.",
            kind="warning",
        )
        break
    elif result["status"] == 403:
        display.banner(
            f"Daily cap hit during the burst at request #{i} (see next section).",
            kind="warning",
        )
        break

display.show_table(burst_log)


In [ ]:
def _to_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

chart_log = [
    {"request_number": r["request_number"], "remaining_tokens": _to_int(r["remaining_tokens"])}
    for r in burst_log
]
display.plot_remaining_tokens(chart_log, trip_index=tripped_at)


## Accumulate to the daily 403

We keep sending requests -- sleeping for `Retry-After` seconds whenever we
hit a 429 so the **TPM** limit isn't what blocks us -- until the cumulative
**daily** token budget is exhausted and the API returns **403**. The loop is
bounded by both a maximum number of iterations and a maximum wall-clock time
so it cannot hang a workshop session.


In [ ]:
MAX_ITERATIONS = 60
MAX_WALL_CLOCK_SECONDS = 180

daily_log = []
daily_403_seen = False
deadline = time.time() + MAX_WALL_CLOCK_SECONDS

for i in range(1, MAX_ITERATIONS + 1):
    if time.time() > deadline:
        display.banner("Reached max wall-clock time before hitting the daily cap.", kind="warning")
        break

    result = call_chat_completion("Give me a one-word synonym for happy.", DEMO_RUN, max_tokens=20)
    entry = {
        "iteration": i,
        "status": result["status"],
        "tokens_consumed": result["tokens_consumed"],
        "remaining_tokens": result["remaining_tokens"],
        "retry_after": result["retry_after"],
    }
    daily_log.append(entry)

    if result["status"] == 403:
        daily_403_seen = True
        display.banner("Daily token budget exhausted -- 403 returned.", kind="error")
        print(result["body"])
        break
    elif result["status"] == 429:
        sleep_for = _to_int(result["retry_after"]) or 5
        time.sleep(sleep_for)
    # status == 200: continue accumulating

display.show_table(daily_log)

total_consumed = sum(_to_int(r["tokens_consumed"]) or 0 for r in daily_log)
display.show_table([{
    "total_tokens_consumed_this_run": total_consumed,
    "configured_daily_cap": DAILY_TOKEN_CAP,
    "daily_403_observed": daily_403_seen,
}])


## Observe / summary

| Behavior observed | Policy element responsible |
| --- | --- |
| Baseline call succeeds with `tokens-consumed`/`remaining-tokens` headers | `azure-openai-token-limit` (headers enabled via `tokens-consumed-header-name` / `remaining-tokens-header-name`) |
| Burst trips a `429` with `Retry-After` | `azure-openai-token-limit`'s `tokens-per-minute` limit and `retry-after-header-name` |
| Sustained use trips a `403` with a clear body | The daily-quota `cache-lookup-value` / `choose` / `return-response` block, gated by the `demo1-daily-token-cap` named value |
| Counters never mix across demos/attendees | The demo-isolated `counter-key` (`subscription id + x-demo-run`) and the dedicated `demo1-token-governance-sub` subscription |


In [ ]:
display.header("Consolidated results")
all_results = []
for r in burst_log:
    all_results.append({"phase": "burst", **r})
for r in daily_log:
    all_results.append({"phase": "daily", **r})
display.show_table(all_results)


## Reset: change the `x-demo-run` suffix

Because the policy's `counter-key` includes `x-demo-run`, regenerating
`DEMO_RUN` and persisting it resets every counter for this demo **instantly**
-- there is no need to wait for the TPM window or the daily window to roll
over. The cell below regenerates the suffix, persists it to `.env`, and
immediately proves the reset by making a follow-up call that succeeds.


In [ ]:
DEMO_RUN = uuid.uuid4().hex[:8]
config.persist_demo_run(DEMO_RUN)
print(f"New DEMO_RUN suffix: {DEMO_RUN}")

reset_check = call_chat_completion("Say hello again, briefly.", DEMO_RUN)
display.show_table([{
    "status": reset_check["status"],
    "tokens_consumed": reset_check["tokens_consumed"],
    "remaining_tokens": reset_check["remaining_tokens"],
    "reply": reset_check["reply"],
}])

if reset_check["status"] == 200:
    display.banner("Reset confirmed: the follow-up call succeeded immediately.", kind="success")
else:
    explain_failure(reset_check)


## Cleanup (optional)

The cell below removes the demo API, subscription, and named values created
by this notebook. It is **idempotent** -- 404s are tolerated, so it is safe
to run even if some or all resources were already removed (or never fully
created). This cell is **not run automatically**; uncomment and run it only
when you are done with Demo 1.


In [ ]:
# Uncomment to clean up Demo 1 resources.
# apim.delete_api_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID)
# apim.delete_subscription_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID)
# apim.delete_named_value_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_NAMED_VALUE_KEY)
# apim.delete_named_value_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_NAMED_VALUE_TPM)
# apim.delete_named_value_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_NAMED_VALUE_DAILY_CAP)
# display.banner("Demo 1 resources removed (idempotent).", kind="success")
